In [1]:
import os
from difflib import SequenceMatcher
from pathlib import Path

import cv2
import numpy as np
from paddleocr import PaddleOCR

d:\MyProgramms\image-detection\.venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.1)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(
d:\MyProgramms\image-detection\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.


In [38]:
ocr = PaddleOCR(use_angle_cls=True, enable_mkldnn=False, lang='ru', ocr_version='PP-OCRv5')

C:\Users\sereg\AppData\Local\Temp\ipykernel_7272\3947172038.py:1: DeprecationWarning: The parameter `use_angle_cls` has been deprecated and will be removed in the future. Please use `use_textline_orientation` instead.
  ocr = PaddleOCR(use_angle_cls=True, enable_mkldnn=False, lang='ru', ocr_version='PP-OCRv5')
Creating model: ('PP-LCNet_x1_0_doc_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\sereg\.paddlex\official_models\PP-LCNet_x1_0_doc_ori`.
Creating model: ('UVDoc', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\sereg\.paddlex\official_models\UVDoc`.
Creating model: ('PP-LCNet_x1_0_textline_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\sereg\.paddlex\official_models\PP-LCNet_x1_0_textline_ori`.
Creating model: ('PP-OCRv5_server_det', None)
Model files already exist

In [54]:

images_dir = Path(r"..\Титры Лунтика Новый 8K Смотреть На ПК и не только")

SIMILARITY_THRESHOLD = 0.85

# Замена латинских символов, похожих на кириллические, для корректного сравнения
LATIN_TO_CYR = str.maketrans("aeopcxylmkivusnrABEHKMOPCTXYLIVUSNR", "аеорсхулмкивуснрАВЕНКМОРСТХУЛИВУСНР")

def normalize(text):
    return text.translate(LATIN_TO_CYR).lower()

prev_text = ""
unique_subtitles = []

for filename in sorted(images_dir.rglob('*'), key=lambda x: (len(str(x)), str(x))):
    if filename.suffix.lower() not in ('.jpg', '.jpeg', '.png'):
        continue

    with open(os.path.join(images_dir, filename), 'rb') as f:
        img = np.asarray(bytearray(f.read()), dtype=np.uint8)
        img = cv2.imdecode(img, cv2.IMREAD_COLOR)

    print(f"Обработка: {filename.name}")

    rec_texts = ocr.predict(img)[0]['rec_texts']
    text = " ".join(rec_texts).strip()

    if not text:
        continue

    norm_text = normalize(text)

    # Пропустить кадр, если он слишком похож на предыдущий
    if SequenceMatcher(None, normalize(prev_text), norm_text).ratio() >= SIMILARITY_THRESHOLD:
        print(f"  (пропущено как дубль)")
        continue

    print(f"Субтитры: {norm_text}")
    unique_subtitles.append(norm_text)
    prev_text = norm_text

print(f"\nУникальных субтитров: {len(unique_subtitles)}")


Обработка: frame_1.jpg
Субтитры: роли озвучивали
Обработка: frame_2.jpg
Субтитры: роли озвучивали екатерина гороховская олег куликович михаил черняк елена шульман
Обработка: frame_3.jpg
  (пропущено как дубль)
Обработка: frame_4.jpg
Субтитры: роли озвучивали екатерина гороховская олег куликович михаил черняк елена шульман автор сценария фёдор дмитриев
Обработка: frame_5.jpg
  (пропущено как дубль)
Обработка: frame_6.jpg
  (пропущено как дубль)
Обработка: frame_7.jpg
Субтитры: роли озвучивали олег куликович екатерина гороховская черняк елена шульман михаил автор сценария фёдор дмитриев режиссер сериала елена галдобина
Обработка: frame_8.jpg
Субтитры: роли озвучивали екатерина гороховская олег куликович михаил черняк елена шульман автор сценария фёдор дмитриев режиссер сериала елена галдобина режиссер серуу екатерина салабай
Обработка: frame_9.jpg
  (пропущено как дубль)
Обработка: frame_10.jpg
Субтитры: слена шульман автор сценария фёдор дмитриев режиссер сериала елена галдобина режиссе

In [ ]:

def merge_subtitles(subtitles):
    """Объединяет субтитры, устраняя перекрывающиеся слова между соседними строками."""
    if not subtitles:
        return ""

    all_words = []

    all_words.extend(subtitles[0].split())
    
    for next_text in subtitles[1:]:
        print(next_text)
        current = next_text.split()
        
        index_from = min(len(current), len(all_words))
        print(zip(all_words[-index_from:], current[:index_from]))
        from_ = 0
        for previous_word, current_word in zip(all_words[-index_from:], current[:index_from]):
            if previous_word != current_word:
                break
            
            from_ += 1

        all_words.extend(current[from_:])
        
    return " ".join(all_words)


full_text = merge_subtitles(unique_subtitles)
print(full_text)


роли озвучивали екатерина гороховская олег куликович михаил черняк елена шульман автор сценария фёдор дмитриев роли озвучивали олег куликович екатерина гороховская черняк елена шульман михаил автор сценария фёдор дмитриев режиссер сериала елена галдобина роли озвучивали екатерина гороховская олег куликович михаил черняк елена шульман автор сценария фёдор дмитриев режиссер сериала елена галдобина режиссер серуу екатерина салабай художник-постановщик марина комаркевич композиторы кошеваров максим михаил чертищев режиссер серии екатерина салабай художник-постановщик марина комаркевич композиторы максим кошеваров михаил чертищев звукорежиссёр е виноградова катерина елена павликова музыкальный редактор валентин васенков монтаж роман смородин ведущий аниматор ольга тарарина консультант бронзит константин аниматик дарина шмидт аниматоры марина зарипова цимбалок ведущий аниматор ольга тарарина консультант константин бронзит аниматик дарина шмидт аниматоры любовь цимбалок марина зарипова елена 